[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/paulnovello/Advanced-AI/blob/main/PP6%3A%SSL/ssl_part1_pretext_tasks.ipynb)
# SSL Part 1 — Pretext Tasks for Self-Supervised Learning

## Why Self-Supervised Learning?

In standard supervised learning, we need **labeled data** — and labels are expensive. Annotating millions of images by hand is slow, costly, and sometimes impossible (e.g., medical imaging).

**Self-supervised learning (SSL)** sidesteps this bottleneck: instead of human-provided labels, we design **pretext tasks** that generate supervision signals *from the data itself*.

The key insight is:

> If a network can solve a well-chosen pretext task, it must have learned **meaningful representations** of the data along the way.

---

### What We Will Do

In this notebook, we will:

1. Implement three classic pretext tasks from scratch: **rotation prediction**, **jigsaw puzzle**, and **colorization**
2. Train a small CNN backbone on each task — **without using any class labels**
3. Evaluate the quality of learned representations via **linear probing**
4. Visualize feature spaces with **t-SNE** to see that semantic structure emerges *without any supervision*

We compare all methods against a **random baseline** (lower bound) and **fully supervised training** (upper bound).

---

### Pretext Tasks at a Glance

| Task | Input | Supervision Signal | What the Network Must Learn |
|---|---|---|---|
| **Rotation** | Rotated image | Which rotation was applied (0°/90°/180°/270°) | Object orientation → semantic understanding |
| **Jigsaw** | Shuffled patches | Which permutation was used | Spatial layout → structural understanding |
| **Colorization** | Grayscale image | Original RGB colors | Object identity → semantic understanding |

The core idea is the same: force the network to solve a task that *requires* understanding image content.

---
##  Setup

In [ ]:
# !pip install -q torch torchvision matplotlib scikit-learn tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from tqdm.auto import tqdm
from tqdm.notebook import tqdm as tqdm_nb
import itertools
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

We will work with **CIFAR-10**, a standard computer vision dataset with 60,000 32×32 color images in 10 classes.

This is a good testbed for SSL experiments: small enough to train quickly, but complex enough that random features won't work well.

In [ ]:
cifar10_train = torchvision.datasets.CIFAR10(root="./data", train=True, download=True)
cifar10_test  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True)

CIFAR10_CLASSES = cifar10_train.classes
print(f"Train: {len(cifar10_train)} images | Test: {len(cifar10_test)} images")
print(f"Classes: {CIFAR10_CLASSES}")
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    img, label = cifar10_train[i]
    ax.imshow(img)
    ax.set_title(f"{CIFAR10_CLASSES[label]}", fontsize=12)
    ax.axis("off")
fig.suptitle("CIFAR-10 sample images", fontsize=14)
plt.tight_layout()
plt.show()

---
## The Backbone

All our experiments share the **same backbone architecture** — a small 3-block CNN that outputs a **256-dimensional feature vector**.

This is critical for fair comparison: since the architecture is identical, any difference in downstream performance is due to the **pretext task** used for training.

The architecture:
- **Block 1:** Conv(3→32) + BatchNorm + ReLU
- **Block 2:** Conv(32→64) + BatchNorm + ReLU + MaxPool
- **Block 3:** Conv(64→128) + BatchNorm + ReLU + MaxPool
- **Head:** AdaptiveAvgPool → Linear(128×4×4, 256)

In [ ]:
# --- Define the shared backbone architecture ---

class SmallConvNet(nn.Module):
    """Small CNN backbone: 3 conv blocks → adaptive pool → 256-d feature vector."""

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 3 → 32 channels
            #...
            # Block 2: 32 → 64 channels + downsample
            #...
            # Block 3: 64 → 128 channels + downsample
            #...
            # Pool to fixed spatial size
            #...

            # Block 3: 64 → 128 channels + downsample
            # ...

            # Pool to fixed spatial size
            nn.AdaptiveAvgPool2d(4),
        )
        self.fc = #...

    def forward(self, x):
       # ...
        return x


# Quick sanity check
_test_backbone = SmallConvNet()
n_params = sum(p.numel() for p in _test_backbone.parameters())
print(f"Backbone parameters: {n_params:,}")
print(f"Output shape: {_test_backbone(torch.randn(1, 3, 32, 32)).shape}")  # [1, 256]

---
## Shared Training & Evaluation Utilities

We define **once** the training loop, hyperparameters, transform, and evaluation tools that every pretext task (and supervised training) will share. Each new task only needs to provide:
1. A **dataset** class (that generates input-label pairs from the pretext task)
2. A **head** on top of a fresh backbone

This modular design mirrors real SSL research: you swap the pretext task, everything else stays the same.

In [ ]:
# --- Shared hyperparameters ---

NUM_EPOCHS = 10
BATCH_SIZE = 256
LR = 1e-3

# --- Shared transform ---

NORMALIZE = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
])

In [ ]:
# --- Shared training loop ---

def train(model, train_loader, criterion, num_epochs=NUM_EPOCHS, lr=LR,
          val_loader=None, task_type="classification", device=device):
    """Unified training loop for classification and regression pretext tasks.

    Args:
        task_type: "classification" (tracks accuracy) or "regression" (tracks val loss).

    Returns:
        history dict with 'train_loss' and 'val_acc' or 'val_loss'.
    """
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": []}
    if task_type == "classification":
        history["val_acc"] = []
    else:
        history["val_loss"] = []

    for epoch in tqdm_nb(range(1, num_epochs + 1), desc="epoch", leave=False):
        # --- Train ---
        model.train()
        total_loss = 0.0
        for inputs, targets in tqdm_nb(train_loader, desc="batch", leave=False):
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = #...
            # compute loss and backpropagate
            # ...
            # ...
            # ...
            optimizer.step()
            total_loss += loss.item() * inputs.size(0)
        avg_loss = total_loss / len(train_loader.dataset)
        history["train_loss"].append(avg_loss)

        # --- Validate ---
        if val_loader is not None:
            model.eval()
            with torch.no_grad():
                if task_type == "classification":
                    correct, total = 0, 0
                    for inputs, targets in val_loader:
                        inputs, targets = inputs.to(device), targets.to(device)
                        preds = model(inputs).argmax(dim=1)
                        correct += (preds == targets).sum().item()
                        total += targets.size(0)
                    val_metric = correct / total
                    history["val_acc"].append(val_metric)
                    print(f"  Epoch {epoch:2d}/{num_epochs} — loss: {avg_loss:.4f} — val acc: {val_metric:.1%}")
                else:
                    val_total = 0.0
                    for inputs, targets in val_loader:
                        inputs, targets = inputs.to(device), targets.to(device)
                        val_total += criterion(model(inputs), targets).item() * inputs.size(0)
                    val_metric = val_total / len(val_loader.dataset)
                    history["val_loss"].append(val_metric)
                    print(f"  Epoch {epoch:2d}/{num_epochs} — train loss: {avg_loss:.4f} — val loss: {val_metric:.4f}")
        else:
            print(f"  Epoch {epoch:2d}/{num_epochs} — loss: {avg_loss:.4f}")

    return history

In [ ]:
# --- Shared plotting ---

def plot_training_curves(history, task_name):
    """Plot training loss and validation metric (accuracy or loss)."""
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(epochs, history["train_loss"], marker="o")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Train loss")
    ax1.set_title(f"{task_name} — Training loss")

    if "val_acc" in history:
        ax2.plot(epochs, [a * 100 for a in history["val_acc"]], marker="o", color="green")
        ax2.set_ylabel("Val accuracy (%)")
        ax2.set_title(f"{task_name} — Validation accuracy")
    else:
        ax2.plot(epochs, history["val_loss"], marker="o", color="orange")
        ax2.set_ylabel("Val loss")
        ax2.set_title(f"{task_name} — Validation loss")
    ax2.set_xlabel("Epoch")

    plt.tight_layout()
    plt.show()

The following code is used to extract features from the backbone and fit a logistic regression on top of them.  
This is the standard procedure to evaluate the quality of the features learned by the backbone.

In [ ]:
# --- Shared evaluation: linear probing, feature extraction, t-SNE ---

@torch.no_grad()
def extract_features(backbone, dataset, device=device, batch_size=512):
    """Pass all images through a frozen backbone and return (features, labels) as numpy arrays."""
    backbone.eval()
    backbone.to(device)

    transform = T.Compose([T.ToTensor(), T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

    all_features, all_labels = [], []
    for i in range(0, len(dataset), batch_size):
        batch_imgs, batch_labels = [], []
        for j in range(i, min(i + batch_size, len(dataset))):
            img, label = dataset[j]
            batch_imgs.append(transform(img))
            batch_labels.append(label)

        imgs = torch.stack(batch_imgs).to(device)
        features = backbone(imgs).cpu().numpy()
        all_features.append(features)
        all_labels.extend(batch_labels)

    return np.concatenate(all_features, axis=0), np.array(all_labels)


def linear_probe(backbone, name, device=device):
    """Freeze backbone, extract features, and fit a logistic regression on CIFAR-10.

    Features are pre-extracted with the backbone in eval mode to avoid
    corrupting BatchNorm running statistics.

    Returns:
        (accuracy, test_feats, test_labels) — features are returned for t-SNE.
    """
    print(f"Extracting features from {name} backbone...")
    train_feats, train_labels = extract_features(backbone, cifar10_train, device)
    test_feats, test_labels   = extract_features(backbone, cifar10_test, device)

    print("Fitting linear probe...")
    clf = LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs", multi_class="multinomial")
    clf.fit(train_feats, train_labels)
    acc = clf.score(test_feats, test_labels)
    print(f"  → {name} linear probe accuracy: {acc:.1%}\n")

    return acc, test_feats, test_labels


def plot_tsne(test_feats, test_labels, title, n_samples=2000):
    """Run t-SNE on backbone features and plot colored scatter + image thumbnails."""
    indices = np.random.choice(len(test_feats), n_samples, replace=False)
    feats_subset = test_feats[indices]
    labels_subset = test_labels[indices]

    print(f"Running t-SNE on {n_samples} samples...")
    tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, n_iter=1000)
    emb = tsne.fit_transform(feats_subset)
    print("Done!")

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))

    cmap = plt.cm.tab10
    for class_idx in range(10):
        mask = labels_subset == class_idx
        axes[0].scatter(emb[mask, 0], emb[mask, 1], c=[cmap(class_idx)],
                        label=CIFAR10_CLASSES[class_idx], s=8, alpha=0.6)
    axes[0].legend(fontsize=9, markerscale=3, loc="best")
    axes[0].set_title(f"t-SNE of {title} features (colored by class)", fontsize=13)
    axes[0].set_xlabel("t-SNE dim 1")
    axes[0].set_ylabel("t-SNE dim 2")

    N_THUMBNAILS = 200
    thumb_indices = np.random.choice(n_samples, N_THUMBNAILS, replace=False)
    axes[1].set_title(f"t-SNE with image thumbnails ({title})", fontsize=13)
    axes[1].set_xlabel("t-SNE dim 1")
    axes[1].set_ylabel("t-SNE dim 2")
    axes[1].scatter(emb[:, 0], emb[:, 1], c="lightgray", s=2, alpha=0.3)

    for ti in thumb_indices:
        global_idx = indices[ti]
        img_pil = cifar10_test[global_idx][0]
        img_np = np.array(img_pil)
        imagebox = OffsetImage(img_np, zoom=0.6)
        ab = AnnotationBbox(imagebox, (emb[ti, 0], emb[ti, 1]), frameon=False, pad=0)
        axes[1].add_artist(ab)

    plt.tight_layout()
    plt.show()

In [ ]:
# --- Helper to build data loaders ---

def make_loaders(train_ds, val_ds, batch_size=BATCH_SIZE):
    """Create train and val DataLoaders with standard settings."""
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
    return train_loader, val_loader

---
## Random Baseline (lower bound)

Before training anything, let's establish a **lower bound**: how well does a completely **untrained** (randomly initialized) backbone perform?

We extract features from a random backbone and run linear probing + t-SNE. This tells us:
- What accuracy we get "for free" from the architecture alone
- What the feature space looks like before any learning

Any SSL method worth its salt should significantly beat this baseline.

In [ ]:
random_backbone = SmallConvNet()  # untrained baseline
# train a linear head on top of the backbone and plot the t-SNE of its features
random_probe_acc, rand_test_feats, rand_test_labels = # ...
# t-sne plot
# ...

---
## Fully Supervised Upper Bound

Before diving into self-supervised pretext tasks, let's establish an **upper bound** by training the same backbone in a fully supervised fashion on CIFAR-10.

This gives us a reference point: how well can this small network do when it has access to **all labels**?

The gap between supervised performance and each pretext task's linear probe accuracy will quantify how much room there is for improvement.


In [ ]:
# Build a supervised model: backbone + ReLU10-class head
supervised_backbone = SmallConvNet()
supervised_model = nn.Sequential(
    #...
).to(device)

supervised_train_ds = torchvision.datasets.CIFAR10(root="./data", train=True,  transform=NORMALIZE)
supervised_test_ds  = torchvision.datasets.CIFAR10(root="./data", train=False, transform=NORMALIZE)
supervised_train_loader, supervised_test_loader = make_loaders(supervised_train_ds, supervised_test_ds)

supervised_history = train(
    # ...
)
supervised_acc = supervised_history["val_acc"][-1]
print(f"\nSupervised test accuracy: {supervised_acc:.1%}")

> **Expected result:** ~75–85% test accuracy. This is the ceiling for our small backbone — all self-supervised methods below will be compared against this upper bound.
>
> Note: state-of-the-art CIFAR-10 uses much larger architectures (ResNets, ViTs). Our small 3-block CNN is intentionally simple to keep training fast — the goal is to compare *learning methods*, not push for SOTA.

---
## Section 2 — Pretext Task 1: Rotation Prediction

**Paper:** *Unsupervised Representation Learning by Predicting Image Rotations* (Gidaris et al., 2018)

**Idea:** Rotate an image by one of {0°, 90°, 180°, 270°} and train the network to predict which rotation was applied.

### Why does this work?

To recognize that an image is upside-down, the network must understand *what* is in the image. For example:
- Cars have wheels on the **bottom**
- Skies are on **top**
- Text reads left-to-right

This forces the backbone to learn **semantic features** — *without ever seeing a single label*.

### The pipeline

$$
\text{Image} \xrightarrow{\text{random rotation}} \text{Rotated image} \xrightarrow{\text{backbone}} \mathbf{z} \in \mathbb{R}^{256} \xrightarrow{\text{linear head}} \hat{y} \in \{0°, 90°, 180°, 270°\}
$$

The loss is standard **cross-entropy** over 4 classes. After training, we **discard the head** and keep only the backbone.

### 2.1 — Rotation dataset

The dataset wrapper is simple:
1. Take an image from CIFAR-10 (ignoring its label)
2. Pick a **random rotation** from {0°, 90°, 180°, 270°}
3. Apply the rotation
4. Return `(rotated_image, rotation_label)`

Note that the original CIFAR-10 class label is **completely discarded** — the network never sees it.

In [ ]:
# --- Rotation dataset ---

ROTATION_ANGLES = {0: 0, 1: 90, 2: 180, 3: 270}


class RotationDataset(Dataset):
    """Wraps a torchvision dataset: returns (rotated_image, rotation_label).

    The original class labels are completely ignored.
    """

    def __init__(self, base_dataset, transform=NORMALIZE):
        self.base_dataset = base_dataset
        self.transform = transform

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        img, _ = self.base_dataset[idx]  # ignore the label!
        # Pick a random rotation
        rot_label = # random between 0 and 3
        angle = # the corresponding angle
        img_rotated = TF.rotate(img, angle)
        if self.transform:
            img_rotated = self.transform(img_rotated)
        return img_rotated, rot_label


rotation_train_ds = RotationDataset(cifar10_train)
rotation_val_ds   = RotationDataset(cifar10_test)

print(f"Rotation train: {len(rotation_train_ds)} | val: {len(rotation_val_ds)}")

In [ ]:
# Check the first few examples to make sure each sample is composed of a rotated image and a rotation label between 0 and 3

In [ ]:
# --- Visualize some rotated examples ---

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    img, rot_label = rotation_train_ds[i]
    # Undo normalization for display
    img_display = img * 0.5 + 0.5
    ax.imshow(img_display.permute(1, 2, 0).numpy())
    ax.set_title(f"{ROTATION_ANGLES[rot_label]}°", fontsize=12)
    ax.axis("off")
fig.suptitle("Rotation prediction — sample inputs", fontsize=14)
plt.tight_layout()
plt.show()

### 2.2 — Rotation model

We attach a simple **linear classification head** (256 → 4) on top of the backbone.

This is intentionally minimal: we want the backbone to do the heavy lifting. After pretraining, we **keep the backbone** and **discard this head**.

In [ ]:
# --- Rotation prediction model ---

# Build a backbone + ReLU + 4-class head
rotation_backbone = SmallConvNet()
rotation_model = nn.Sequential(
    #...
).to(device)
print("Rotation model ready.")

### 2.3 — Training

We train for 10 epochs using cross-entropy loss. Remember: **no CIFAR-10 class labels are used** — the only supervision comes from the rotation angle (0, 1, 2, or 3).

This is the essence of self-supervised learning: the labels are *generated automatically* from the data transformation.

In [ ]:
# --- Train rotation prediction ---

rotation_train_loader, rotation_val_loader = make_loaders(rotation_train_ds, rotation_val_ds)

rotation_history = train(
    # ...
)
print(f"\nFinal rotation val accuracy: {rotation_history['val_acc'][-1]:.1%}")

> **Expected result:** ~75–85% rotation accuracy. If your result is significantly lower, check that the rotations are being applied correctly.

### 2.4 — Evaluation

Now comes the key question: **did the backbone learn useful features?**

Rotation accuracy tells us the *head* is working, but we care about what the *backbone* learned about image content. We evaluate this with two tools:

#### Linear Probing

We **freeze** the pretrained backbone and train a simple logistic regression on top of the extracted features to classify CIFAR-10 classes. High accuracy means the backbone has learned features that are linearly separable by semantic class — a strong signal that the representations are useful.

#### t-SNE Visualization

We project the 256-d feature vectors into 2D using t-SNE. If the pretext task learned meaningful features, we should see **clusters** corresponding to CIFAR-10 classes — even though the backbone never saw those classes during training.

In [ ]:
plot_training_curves(rotation_history, "Rotation")

In [ ]:
# train a linear head on top of the backbone and plot the t-SNE of its features
rotation_probe_acc, rot_test_feats, rot_test_labels = # ...
# t-sne plot
# ...

### What to observe

- **Linear probe accuracy** should be significantly higher than the random baseline — the backbone has learned something useful
- **t-SNE plot** should show emerging clusters by CIFAR-10 class, even though the backbone was never trained on those classes
- Compare the t-SNE scatter with the random baseline: you should see much more structure

The rotation pretext task works because recognizing orientation requires understanding object semantics. A network that knows "this is upside-down" implicitly knows *what* the object is.

---
## Section 3 — Pretext Task 2: Jigsaw Puzzle

**Paper:** *Unsupervised Visual Representation Learning by Context Prediction* (Noroozi & Favaro, 2016)

**Idea:** Split the image into a grid of patches, shuffle them, and train the network to predict which permutation was applied.

We use a **2×2 grid** (4 patches of 16×16 pixels) to keep things manageable at CIFAR-10's 32×32 resolution. This gives us $4! = 24$ possible permutations.

### Why does this work?

To reassemble a puzzle, the network must understand **spatial relationships** between different parts of the image:
- "The sky goes on top, the ground goes on the bottom"
- "The left side of a car's hood connects to the right side"
- "Eyes go above the mouth"

This complements rotation prediction: while rotation captures **global orientation**, jigsaw captures **local spatial structure**.

### The pipeline

$$
\text{Image} \xrightarrow{\text{split 2×2}} 4 \text{ patches} \xrightarrow{\text{shuffle}} \text{Reassembled image} \xrightarrow{\text{backbone + head}} \hat{y} \in \{0, \ldots, 23\}
$$

### 3.1 — Jigsaw dataset

The dataset:
1. Splits each image into a **2×2 grid** of 16×16 patches
2. Picks a **random permutation** from all 24 possibilities
3. Reassembles the image with shuffled patches
4. Returns `(shuffled_image, permutation_index)`

Again, the original class labels are ignored.

In [ ]:
# --- Jigsaw dataset ---

# All 24 permutations of 4 patches (0=top-left, 1=top-right, 2=bottom-left, 3=bottom-right)
ALL_PERMUTATIONS = list(itertools.permutations(range(4)))
NUM_PERMUTATIONS = len(ALL_PERMUTATIONS)  # 24
print(f"Number of permutations: {NUM_PERMUTATIONS}")


class JigsawDataset(Dataset):
    """Wraps a torchvision dataset: splits image into 2x2 patches,
    shuffles them, and returns (shuffled_image, permutation_index).

    Original class labels are ignored.
    """

    def __init__(self, base_dataset, transform=NORMALIZE):
        self.base_dataset = base_dataset
        self.transform = transform

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        img, _ = self.base_dataset[idx]  # ignore the label
        # Convert PIL to tensor for slicing
        img_tensor = T.ToTensor()(img)  # [3, 32, 32]

        # Split into 2x2 grid of 16x16 patches
        patches = [
            img_tensor[:, :16, :16],   # top-left
            img_tensor[:, :16, 16:],   # top-right
            img_tensor[:, 16:, :16],   # bottom-left
            img_tensor[:, 16:, 16:],   # bottom-right
        ]

        # Pick a random permutation
        perm_idx = np.random.randint(0, NUM_PERMUTATIONS)
        perm = ALL_PERMUTATIONS[perm_idx]

        # Reassemble image with shuffled patches
        shuffled = torch.zeros_like(img_tensor)
        shuffled[:, :16, :16]  = patches[perm[0]]
        shuffled[:, :16, 16:]  = patches[perm[1]]
        shuffled[:, 16:, :16]  = patches[perm[2]]
        shuffled[:, 16:, 16:]  = patches[perm[3]]

        # Normalize
        shuffled = T.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))(shuffled)

        return shuffled, perm_idx


jigsaw_train_ds = JigsawDataset(cifar10_train)
jigsaw_val_ds   = JigsawDataset(cifar10_test)

print(f"Jigsaw train: {len(jigsaw_train_ds)} | val: {len(jigsaw_val_ds)}")

In [ ]:
# --- Visualize jigsaw examples: original vs shuffled ---

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    # Original
    orig_img = T.ToTensor()(cifar10_train[i][0])
    axes[0, i].imshow(orig_img.permute(1, 2, 0).numpy())
    axes[0, i].set_title("Original" if i == 0 else "", fontsize=11)
    axes[0, i].axis("off")

    # Shuffled
    shuffled_img, perm_idx = jigsaw_train_ds[i]
    shuffled_display = shuffled_img * 0.5 + 0.5
    axes[1, i].imshow(shuffled_display.permute(1, 2, 0).numpy())
    axes[1, i].set_title(f"Shuffled" if i == 0 else "", fontsize=11)
    axes[1, i].axis("off")

fig.suptitle("Jigsaw puzzle — original (top) vs. shuffled (bottom)", fontsize=14)
plt.tight_layout()
plt.show()

### 3.2 — Jigsaw model

Same structure as rotation: backbone + linear head. But now the head predicts **24 classes** (one per permutation) instead of 4.

> **Important:** We use a **fresh backbone** — not the one pretrained on rotation. Each pretext task gets its own randomly initialized backbone so we can compare fairly.

In [ ]:
# --- Jigsaw prediction model ---

# IMPORTANT: fresh backbone — don't reuse the rotation one!
# Build a backbone + ReLU + 24-class head
jigsaw_backbone = SmallConvNet()
jigsaw_model = nn.Sequential(
    #...
).to(device)
print("Jigsaw model ready (fresh backbone).")

### 3.3 — Training

Same setup as rotation: cross-entropy loss, 10 epochs, no class labels.

In [ ]:
# --- Train jigsaw prediction ---

jigsaw_train_loader, jigsaw_val_loader = make_loaders(jigsaw_train_ds, jigsaw_val_ds)

jigsaw_history = train(
    # ...
)
print(f"\nFinal jigsaw val accuracy: {jigsaw_history['val_acc'][-1]:.1%}")

> **Expected result:** ~30–45% jigsaw accuracy. This is much lower than rotation — but remember, the task is much harder (24 classes vs. 4). Random chance is only $\frac{1}{24} \approx 4.2\%$, so 30–45% is actually a strong result.

### 3.4 — Results

In [ ]:
plot_training_curves(jigsaw_history, "Jigsaw")

In [ ]:
# train a linear head on top of the backbone and plot the t-SNE of its features
jigsaw_probe_acc, jig_test_feats, jig_test_labels = # ...
# t-sne plot
# ...

### What to observe

Lower pretext accuracy doesn't necessarily mean worse representations. The jigsaw task is inherently harder (24 classes vs. 4), and the model must learn to understand **spatial layout** to solve it.

Compare the linear probe accuracy and t-SNE plots between rotation and jigsaw:
- Are the clusters similarly structured?
- Does one method produce tighter class separation?

The key insight: **pretext task difficulty and downstream quality are different things**. A task can be hard to solve perfectly but still force the backbone to learn excellent features.

---
## Bonus — Pretext Task 3: Colorization

**Paper:** *Colorful Image Colorization* (Zhang et al., 2016)

**Idea:** Convert an image to grayscale and train the network to predict the original colors.

### Why does this work?

To predict plausible colors, the network must understand *what* objects are present:
- Sky → blue
- Grass → green
- Fire trucks → red

This requires **semantic understanding** at a pixel level — the network can't just memorize color patterns, it needs to recognize objects.

### Key differences from rotation/jigsaw

| | Rotation / Jigsaw | Colorization |
|---|---|---|
| **Task type** | Classification | Regression |
| **Loss** | Cross-entropy | MSE |
| **Output** | Class index | 3-channel RGB image |
| **Input channels** | 3 (RGB) | 1 (grayscale, replicated to 3) |

### The pipeline

$$
\text{RGB Image} \xrightarrow{\text{grayscale}} \text{1-channel input} \xrightarrow{\text{backbone}} \mathbf{z} \in \mathbb{R}^{256} \xrightarrow{\text{decoder head}} \hat{\mathbf{I}} \in \mathbb{R}^{3 \times 32 \times 32}
$$

The decoder head maps the 256-d feature vector back to a full-resolution 3-channel image using linear layers + `Tanh` (to match the $[-1, 1]$ normalization range).

In [ ]:
# --- Colorization dataset ---

class ColorizationDataset(Dataset):
    """Returns (grayscale_image, color_image) pairs for colorization.

    Input: 1-channel grayscale (replicated to 3 channels for the backbone).
    Target: 3-channel normalized color image.
    """

    def __init__(self, base_dataset):
        self.base_dataset = base_dataset

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        img, _ = self.base_dataset[idx]
        color = T.ToTensor()(img)  # [3, 32, 32]
        gray = T.Grayscale(num_output_channels=3)(img)  # PIL grayscale → 3-channel
        gray = T.ToTensor()(gray)
        # Normalize both
        color = T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))(color)
        gray = T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))(gray)
        return gray, color


color_train_ds = ColorizationDataset(cifar10_train)
color_val_ds   = ColorizationDataset(cifar10_test)

### Colorization dataset

The dataset converts each image to grayscale (replicated to 3 channels so the backbone architecture stays the same) and pairs it with the original color image as the target.

In [ ]:
# --- Visualize colorization pairs ---

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    gray, color = color_train_ds[i]
    axes[0, i].imshow((gray * 0.5 + 0.5).permute(1, 2, 0).numpy())
    axes[0, i].set_title("Input" if i == 0 else "", fontsize=11)
    axes[0, i].axis("off")
    axes[1, i].imshow((color * 0.5 + 0.5).permute(1, 2, 0).numpy().clip(0, 1))
    axes[1, i].set_title("Target" if i == 0 else "", fontsize=11)
    axes[1, i].axis("off")
fig.suptitle("Colorization — grayscale input (top) vs. color target (bottom)", fontsize=14)
plt.tight_layout()
plt.show()

### Colorization model

The model architecture is slightly different here. Since we need to reconstruct an entire image (not predict a class), the head is a small **decoder**:

`backbone (256-d)` → `Linear(256, 512)` → `ReLU` → `Linear(512, 3×32×32)` → `Tanh` → `Reshape`

> Note: this is a very simple decoder (just linear layers). Real colorization models use U-Net or decoder CNNs for much better results. Our goal here is just to evaluate what the **backbone** learns.

In [ ]:
# --- Colorization model ---

class Reshape(nn.Module):
    """Reshape layer for use inside nn.Sequential."""
    def __init__(self, *shape):
        super().__init__()
        self.shape = shape
    def forward(self, x):
        return x.view(x.size(0), *self.shape)

color_backbone = SmallConvNet()
color_model = nn.Sequential(
    color_backbone,
    nn.ReLU(),
    nn.Linear(256, 512),
    nn.ReLU(inplace=True),
    nn.Linear(512, 3 * 32 * 32),
    nn.Tanh(),            # output in [-1, 1] to match normalized images
    Reshape(3, 32, 32),
).to(device)
print("Colorization model ready (fresh backbone).")

In [ ]:
# --- Train colorization ---

color_train_loader, color_val_loader = make_loaders(color_train_ds, color_val_ds)

color_history = train(
    color_model, color_train_loader, nn.MSELoss(),
    val_loader=color_val_loader, task_type="regression",
)

### Training

We train with **MSE loss** (regression) instead of cross-entropy (classification). The validation metric is now validation loss rather than accuracy.

In [ ]:
# --- Show colorization predictions ---

color_model.eval()
fig, axes = plt.subplots(3, 8, figsize=(16, 6))

with torch.no_grad():
    for i in range(8):
        gray, color_target = color_val_ds[i]
        pred = color_model(gray.unsqueeze(0).to(device)).cpu().squeeze(0)

        axes[0, i].imshow((gray * 0.5 + 0.5).permute(1, 2, 0).numpy())
        axes[0, i].set_title("Input" if i == 0 else "", fontsize=11)
        axes[0, i].axis("off")

        axes[1, i].imshow((pred * 0.5 + 0.5).permute(1, 2, 0).numpy().clip(0, 1))
        axes[1, i].set_title("Predicted" if i == 0 else "", fontsize=11)
        axes[1, i].axis("off")

        axes[2, i].imshow((color_target * 0.5 + 0.5).permute(1, 2, 0).numpy().clip(0, 1))
        axes[2, i].set_title("Ground truth" if i == 0 else "", fontsize=11)
        axes[2, i].axis("off")

fig.suptitle("Colorization results — input (top) / predicted (mid) / ground truth (bottom)", fontsize=14)
plt.tight_layout()
plt.show()

### Colorization predictions

Let's visualize what the model predicts. Don't expect photorealistic results — with a linear decoder on 32×32 images, the colorizations will be blurry and approximate. What matters is whether the backbone learned useful features for downstream tasks.

In [ ]:
plot_training_curves(color_history, "Colorization")

### Colorization evaluation

Same evaluation protocol: freeze the backbone, extract features, run linear probing and t-SNE.

In [ ]:
color_probe_acc, col_test_feats, col_test_labels = linear_probe(color_backbone, "Colorization")
plot_tsne(col_test_feats, col_test_labels, "colorization-pretrained")

---
## Final Summary

Let's compare all methods side by side. The summary table below shows:
- **Pretext metric:** how well the model solves its pretext task (accuracy for classification, loss for regression)
- **Linear probe:** how useful the learned features are for the actual downstream task (CIFAR-10 classification)

In [ ]:
# --- Final summary table ---

print("\n" + "=" * 65)
print(f"{'Method':<25} {'Pretext Metric':>15} {'Linear Probe':>20}")
print("-" * 65)
print(f"{'Supervised (upper bound)':<25} {'—':>15} {supervised_acc:>19.1%}")
print(f"{'Random (baseline)':<25} {'—':>15} {random_probe_acc:>19.1%}")
print(f"{'Rotation pred.':<25} {rotation_history['val_acc'][-1]:>14.1%} {rotation_probe_acc:>19.1%}")
print(f"{'Jigsaw puzzle':<25} {jigsaw_history['val_acc'][-1]:>14.1%} {jigsaw_probe_acc:>19.1%}")
print(f"{'Colorization':<25} {color_history['val_loss'][-1]:>13.4f}L {color_probe_acc:>19.1%}")
print("=" * 65)

### Key Takeaways

1. **All pretext tasks learn useful features** — they significantly outperform the random baseline on linear probing, proving that meaningful representations emerge *without any class labels*

2. **Lower pretext accuracy ≠ worse features** — jigsaw has lower pretext accuracy than rotation but can yield comparable linear-probe performance. The difficulty of the pretext task and the quality of learned features are orthogonal.

3. **Different tasks capture different aspects** — rotation learns global orientation cues, jigsaw learns spatial structure, and colorization learns object-color associations. Each provides a different inductive bias.

4. **The gap to supervised learning is still large** — fully supervised training reaches ~80–85%, while our best pretext-task features reach ~50–55%. This gap motivated the next generation of SSL methods.

---

### What Comes Next?

These pretext tasks were the **first generation** of visual SSL (2016–2018). Their main limitation is that the pretext task must be carefully designed by hand — and the features learned are often biased toward the specific task (e.g., rotation features are sensitive to orientation but may ignore texture).

The **second generation** — **contrastive learning** (SimCLR, MoCo, BYOL) — addresses this by learning representations that are **invariant to data augmentations** rather than solving a specific pretext task. We'll explore this in the next notebook.